# OPRB benchmark: Tesseract v5.5 (pytesseract, 16-thread CPU)

Local reproducible test-set evaluation using Tesseract 5.5 (`--oem 3 --psm 3`, `OMP_THREAD_LIMIT=16`).

In [ ]:
import os

os.environ["OMP_THREAD_LIMIT"] = "16"
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["OMP_DYNAMIC"] = "FALSE"
TESSERACT_DIR = r"C:\Program Files\Tesseract-OCR"
os.environ["PATH"] = TESSERACT_DIR + os.pathsep + os.environ.get("PATH", "")

import json, re, time, unicodedata
from collections import Counter
from pathlib import Path
import pandas as pd
import pytesseract
from jiwer import cer, wer
from PIL import Image
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pytesseract.pytesseract.tesseract_cmd = os.path.join(TESSERACT_DIR, "tesseract.exe")
DATA_ROOT = ROOT / "test"
OUTPUT_DIR = ROOT / "result" / "tesseract_pytesseract_cpu16"
RESULTS_CSV = OUTPUT_DIR / "per_image_results.csv"
SAVE_EVERY = 50
TEXT_LIMIT = 2000
NONTEXT_LIMIT = 1000
MODEL_ID = "Tesseract 5.5.3.20260724 (pytesseract, eng, OEM 3, PSM 3)"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_gt(path: Path) -> str:
    tokens = []
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        for line in f:
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) >= 2 and fields[0].strip():
                tokens.append(fields[0].strip())
    return " ".join(tokens)

def clean(text: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text)).strip().casefold()

def metrics(reference: str, prediction: str) -> dict:
    ref, hyp = clean(reference), clean(prediction)
    if not ref:
        raise ValueError("Empty GT transcript")
    a, b = Counter(ref.split()), Counter(hyp.split())
    matched = sum((a & b).values())
    precision = matched / sum(b.values()) if b else 0.0
    recall = matched / sum(a.values()) if a else 0.0
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": precision,
        "token_recall": recall,
        "token_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }

def collect_pairs():
    pairs = []
    for category, limit in (("text", TEXT_LIMIT), ("nontext", NONTEXT_LIMIT)):
        image_dir, gt_dir = DATA_ROOT / category / "img", DATA_ROOT / category / "gt"
        images = {p.stem: p for p in image_dir.glob("*.jpg")}
        gts = {p.stem: p for p in gt_dir.glob("*.txt")}
        common = sorted(images.keys() & gts.keys())[:limit]
        for stem in common:
            parts = stem.split("__", 2)
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(images[stem]),
                "gt_path": str(gts[stem]),
                "occlusion_type": parts[0],
                "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
            })
    return pairs

pairs = collect_pairs()
print(f"Pairs: {len(pairs):,}")

In [ ]:
rows = []
t0 = time.perf_counter()

for idx, sample in enumerate(tqdm(pairs, desc="Tesseract OPRB CPU (16-thread limit)"), 1):
    row = {**sample, "model_id": MODEL_ID}
    try:
        start = time.perf_counter()
        with Image.open(sample["image_path"]) as image:
            prediction = pytesseract.image_to_string(image, lang="eng", config="--oem 3 --psm 3")
        latency = time.perf_counter() - start
        reference = read_gt(Path(sample["gt_path"]))
        row.update({
            "status": "ok",
            "prediction": prediction.strip(),
            "ground_truth": reference,
            "latency_sec": latency,
            "generated_tokens": 0,
            "possibly_truncated": False,
            **metrics(reference, prediction),
        })
    except Exception as exc:
        row.update({"status": "error", "error": repr(exc)})
    rows.append(row)
    if idx % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)

pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
ok = pd.DataFrame(rows)
ok = ok.loc[ok["status"] == "ok"]
summary = {
    "model": MODEL_ID,
    "dataset": "kpurkayastha/OPRB test",
    "total_expected": len(pairs),
    "n_success": int(len(ok)),
    "n_errors": int(len(rows) - len(ok)),
    "mean_latency_sec": float(ok["latency_sec"].mean()),
    "mean_cer": float(ok["cer"].mean()),
    "mean_wer": float(ok["wer"].mean()),
    "mean_token_f1": float(ok["token_f1"].mean()),
    "thread_limit": 16,
    "runtime_sec_this_run": time.perf_counter() - t0,
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))